# Catalog / Collection Lifecycle — STAC-native

This notebook walks the full lifecycle through the STAC API:

1. Create a catalog
2. Create a collection with inline schema / layer_config / write_policy
3. Localized update (en + es)
4. Round-trip the collection
5. Soft-delete and assert `collection_configs` rows are cleaned up
6. Zero-config variant — collection created with only id + title, every sub-config resolves via the waterfall

Every sub-config has a code-level default. The waterfall only ever raises
`ConfigResolutionError` (HTTP 500) when a required field is missing at every
scope — a deploy misconfiguration, never a user-visible 4xx.

In [ ]:
import httpx
import os

BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-lifecycle"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

r = await client.post(
    "/stac/catalogs",
    json={
        "id": CATALOG_ID,
        "title": "Lifecycle Demo",
        "description": "STAC-native catalog/collection lifecycle demo.",
    },
)
_check(r, "Catalog")

## 1. Full-form collection — schema, layer_config, write_policy inline

STAC-native inline workflow: everything needed to drive ingestion is submitted
in the creation payload. The service persists each sub-config into its own
`collection_configs` row via the shared applier handlers.

In [ ]:
COLL_FULL = "sensor-stations"

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    json={
        "id": COLL_FULL,
        "title": "Sensor Stations",
        "description": "Point sensor stations with inline schema + write policy.",
        "license": "CC-BY-4.0",
        "extent": {
            "spatial": {"bbox": [[-180, -90, 180, 90]]},
            "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
        },
        "schema": {
            "fields": {
                "geom": {"data_type": "geometry"},
                "station_code": {"data_type": "text", "required": True},
                "reading": {"data_type": "float"},
            },
            "constraints": [
                {"constraint_type": "identity_key", "geohash_precision": 9},
                {"constraint_type": "required", "field": "station_code"},
            ],
        },
        "collection:write_policy": {
            "on_conflict": "update",
            "external_id_field": "properties.station_code",
        },
    },
)
_check(r, "Full-form collection")

## 2. Localized update — Spanish title/description

Collection metadata is internationalised. The update merges the new language
variant without losing existing locales.

In [ ]:
r = await client.put(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_FULL}",
    json={
        "title": {"en": "Sensor Stations", "es": "Estaciones de sensores"},
        "description": {
            "en": "Point sensor stations with inline schema + write policy.",
            "es": "Estaciones puntuales de sensores con esquema y política de escritura.",
        },
    },
)
_check(r, "Localized update")

## 3. Round-trip: GET the collection

The GET response is driver-independent — it comes through the metadata router
(`MetadataRoutingConfig.operations[READ]`), which defaults to Elasticsearch if
available and falls back to Postgres otherwise.

In [ ]:
r = await client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_FULL}",
    params={"lang": "es"},
)
_check(r, "GET (es)")
print("title_es =", r.json().get("title"))

r = await client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_FULL}",
    params={"lang": "en"},
)
_check(r, "GET (en)")
print("title_en =", r.json().get("title"))

## 4. Soft-delete — `collection_configs` must be cleaned

A soft-deleted collection is hidden from the API but its thin registry row is
kept for audit. Every `collection_configs` row bound to the deleted collection
is purged immediately: configs are bound to the *logical* collection identity,
not the physical table, so they would leak state into any future collection
reusing the same id.

In [ ]:
# Drop a transient config right before deletion so we can prove cleanup.
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_FULL}/classes/collection:write_policy",
    json={"on_conflict": "refuse"},
)
_check(r, "Transient write_policy")

r = await client.delete(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_FULL}",
)
_check(r, "Soft-delete")

# A new collection with the same id must not inherit the old policy.
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    json={"id": COLL_FULL, "title": "Recreated Sensors"},
)
_check(r, "Recreate same id")

r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_FULL}/classes/collection:write_policy/effective",
)
print("Effective policy after recreation:", r.json() if r.status_code < 400 else r.text[:200])

## 5. Zero-config collection

Everything is optional. `POST /stac/catalogs/{cat}/collections` with only `id`
and `title` succeeds because every sub-config resolves via the waterfall:

* `CollectionRoutingConfig` → code default `[ItemsPostgresqlDriver]`
* `CollectionWritePolicy` → code default `on_conflict=update`
* `CollectionSchema` → code default — empty fields, no constraints

Ingestion and retrieval work end-to-end without a single config PUT.

In [ ]:
COLL_MIN = "minimal-collection"

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    json={"id": COLL_MIN, "title": "Minimal"},
)
_check(r, "Minimal collection")

feature = {
    "type": "Feature",
    "id": "f-0001",
    "geometry": {"type": "Point", "coordinates": [0, 0]},
    "properties": {"label": "origin"},
}
r = await client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLL_MIN}/items",
    json=feature,
)
_check(r, "Feature ingest")

r = await client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLL_MIN}/items",
)
_check(r, "Feature list")
print("feature count:", len(r.json().get("features", [])))

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()